In [ ]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm
# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

mandl_experiments = [
    ("mumford1", 0.0, 0.5,  0.0),
    ("mumford1", 0.0, 0.5,  0.2),
    ("mumford1", 0.0, 0.5,  0.4),
    ("mumford1", 0.0, 0.5,  0.6),
    ("mumford1", 0.0, 0.5,  0.8),
    ("mumford1", 0.0, 0.5,  1.0),
]


# CSV файл
results_file = Path("experiments_csv/var_conn_coef_small_routes.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity", "seed",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

for dataset_name, dt, rt, ct in tqdm(mandl_experiments, desc="Evaluating mandl"):
    for seed in range (0, 10):
        try:
            run_name = f"mandl_eval_pp_{dt}_op_{rt}_cp_{ct}"

            with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
                cfg_eval = compose(
                    config_name="eval_model_mumford",
                    overrides=[
                        f"+eval={dataset_name}",
                        f"++eval.n_routes=3",
                        f"++eval.min_route_len=3",
                        f"++eval.max_route_len=10",
                        f"+model.weights={model_weights_path}",
                        f"++run_name={run_name}",
                        f"++experiment.seed={seed}",
                        f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                        f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                        f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    ]
                )

            metrics, unserved_demand = main_eval(cfg_eval)
            keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
            row = [dataset_name, dt, rt, ct, seed] + [round(metrics[k].item(), 4) for k in keys_order]

            with open(results_file, mode='a', newline='') as f:
                csv.writer(f).writerow(row)

        except Exception as e:
            print(f"[✗] Failed eval for {run_name}: {e}")

Evaluating mandl: 100%|██████████| 6/6 [09:55<00:00, 99.20s/it] 


In [5]:
from simulation import citygraph_dataset
# from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

In [10]:
dataset = citygraph_dataset.DynamicCityGraphDataset(
    min_nodes=70,
    max_nodes=70,
    edge_keep_prob=0.7,
    data_type=citygraph_dataset.MIXED,  # or any other type you want
    directed=False,
    fully_connected_demand=True,  # default SIDE_LENGTH_M
    mumford_style=True,
    pos_only=False
)

# Generate graphs
n_graphs = 100 # number of graphs you want to generate
graphs = [dataset.generate_graph(draw=False) for _ in tqdm(range(n_graphs))]

100%|██████████| 100/100 [00:03<00:00, 26.66it/s]


In [11]:
import pickle
from pathlib import Path

# Путь для сохранения
save_path = Path('./output_graphs')
if not save_path.exists():
    save_path.mkdir(parents=True)

# Сохраняем объект в файл
with open(save_path / 'raw_graphs_1000.pkl', 'wb') as ff:
    pickle.dump(graphs, ff)


In [12]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm
import torch

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

# Флаг для выбора режима вывода метрик
AVERAGE_METRICS = False  # False для индивидуальных значений, True для усредненных

# Эксперименты для сравнения двух методов
comparison_experiments = [
    ("mumford1", 0.5, 0.5, 0.0),  # Метод 1: акцент на demand_time
    ("mumford1", 0.33, 0.33, 0.33),  # Метод 2: акцент на connectivity
]

# CSV файл с новым именем
results_file = Path("experiments_csv/method_comparison_random_graphs_equalized_coefs.csv")

# Создание заголовка CSV в зависимости от режима
def create_csv_header():
    base_headers = ["dataset", "method", "demand_time", "route_time", "connectivity"]
    metric_headers = ["ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"]
    
    if AVERAGE_METRICS:
        # Для усредненных метрик - один заголовок на метрику
        return base_headers + metric_headers
    else:
        # Для индивидуальных метрик - добавляем индекс элемента
        return base_headers + ["element_idx"] + metric_headers

# Инициализация CSV файла
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow(create_csv_header())

def process_metrics(metrics, keys_order):
    """Обработка метрик в зависимости от их типа"""
    rows_data = []
    
    if AVERAGE_METRICS:
        # Если метрики усреднены (скалярные значения)
        if all(isinstance(metrics[k], torch.Tensor) and metrics[k].numel() == 1 for k in keys_order):
            metric_values = [round(metrics[k].item(), 4) for k in keys_order]
            rows_data.append(metric_values)
        else:
            # Если получили массив, но ожидали усредненные - вычисляем среднее
            metric_values = [round(metrics[k].mean().item(), 4) for k in keys_order]
            rows_data.append(metric_values)
    else:
        # Если метрики не усреднены (массивы значений)
        if all(isinstance(metrics[k], torch.Tensor) for k in keys_order):
            # Определяем количество элементов
            n_elements = max(metrics[k].numel() for k in keys_order)
            
            if n_elements == 1:
                # Если только один элемент
                metric_values = [0] + [round(metrics[k].item(), 4) for k in keys_order]
                rows_data.append(metric_values)
            else:
                # Если несколько элементов - создаем строку для каждого
                for i in range(n_elements):
                    metric_values = [i]  # индекс элемента
                    for k in keys_order:
                        if metrics[k].numel() > 1:
                            metric_values.append(round(metrics[k][i].item(), 4))
                        else:
                            metric_values.append(round(metrics[k].item(), 4))
                    rows_data.append(metric_values)
        else:
            # Если метрики уже скалярные (например, float)
            metric_values = [0] + [round(float(metrics[k]), 4) for k in keys_order]
            rows_data.append(metric_values)
    
    return rows_data

for dataset_name, dt, rt, ct in tqdm(comparison_experiments, desc="Running experiments"):
    method_name = f"Method_DT{dt}_RT{rt}_CT{ct}"
    
    try:
        run_name = f"comparison_exp_{method_name}"

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name}",
                    f"++eval.n_routes=3",
                    f"++eval.min_route_len=3",
                    f"++eval.max_route_len=10",
                    f"++eval.csv=true",  # Включаем CSV режим
                    f"++eval.average_metrics={AVERAGE_METRICS}",  # Устанавливаем флаг усреднения
                    f"+model.weights={model_weights_path}",
                    f"++run_name={run_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                ]
            )

        metrics, unserved_demand = main_eval(cfg_eval)
        keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        
        # Обработка метрик
        metrics_rows = process_metrics(metrics, keys_order)
        
        # Запись в CSV
        with open(results_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            for metric_values in metrics_rows:
                base_row = [dataset_name, method_name, dt, rt, ct]
                full_row = base_row + metric_values
                writer.writerow(full_row)

    except Exception as e:
        print(f"[✗] Failed eval for {run_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"Эксперимент завершен. Результаты сохранены в {results_file}")
print(f"Режим метрик: {'Усредненные' if AVERAGE_METRICS else 'Индивидуальные'}")

Running experiments:   0%|          | 0/2 [00:00<?, ?it/s]Processing...
Done!
Running experiments: 100%|██████████| 2/2 [47:30<00:00, 1425.12s/it]

Эксперимент завершен. Результаты сохранены в experiments_csv/method_comparison_random_graphs_equalized_coefs.csv
Режим метрик: Индивидуальные


In [6]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()
model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

# Два эксперимента
experiments = [
    ("Route_Time_Focus", "mumford1", 1.0, 0.5, 0.0),  # Фокус на времени
    ("Conn_Focus", "mumford1", 0.0, 0.5, 1.0)  # Фокус на связности
]

# CSV файл
results_file = Path("experiments_csv/time_vs_connectivity_comparison.csv")
results_file.parent.mkdir(exist_ok=True)

if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "experiment", "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

for exp_name, dataset, dt, rt, ct in experiments:
    print(f"🔬 Запуск: {exp_name} (dt={dt}, rt={rt}, ct={ct})")
    
    try:
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset}",
                    f"++eval.n_routes=3",
                    f"++eval.min_route_len=3",
                    f"++eval.max_route_len=10",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={exp_name.lower()}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                ]
            )
        
        metrics, unserved_demand = main_eval(cfg_eval)
        keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [exp_name, dataset, dt, rt, ct] + [round(metrics[k].item(), 4) for k in keys_order]
        
        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)
        
        print(f"  ✅ ATT: {metrics['ATT'].item():.2f}, Connectivity: {metrics['median_connectivity'].item():.3f}")
        print(f'userved demand sum: {unserved_demand.mean()}')
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")

print(f"\n📊 Результаты в: {results_file}")

🔬 Запуск: Route_Time_Focus (dt=1.0, rt=0.5, ct=0.0)


Traceback (most recent call last):
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1609, in _pydevd_bundle.pydevd_cython.handle_exception
  File "/root/TNDP_learning/venv/lib/python3.10/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2188, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
  File "/root/TNDP_learning/venv/lib/python3.10/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2257, in _do_wait_suspend
    notify_event.wait(wait_timeout)
  File "/usr/lib/python3.10/threading.py", line 607, in wait
    signaled = self._cond.wait(timeout)
  File "/usr/lib/python3.10/threading.py", line 324, in wait
    gotit = waiter.acquire(True, timeout)
KeyboardInterrupt


  ❌ Ошибка: a Tensor with 2 elements cannot be converted to Scalar
🔬 Запуск: Conn_Focus (dt=0.0, rt=0.5, ct=1.0)
  ❌ Ошибка: a Tensor with 2 elements cannot be converted to Scalar

📊 Результаты в: experiments_csv/time_vs_connectivity_comparison.csv


In [9]:
import os
import shutil

def longest_common_prefix(a: str, b: str) -> str:
    """Возвращает наибольший общий префикс между двумя строками (без расширения)."""
    a_name, _ = os.path.splitext(a)
    b_name, _ = os.path.splitext(b)
    i = 0
    while i < len(a_name) and i < len(b_name) and a_name[i] == b_name[i]:
        i += 1
    return a_name[:i]

def sort_files_into_folders(directory: str, min_prefix_len: int = 3):
    """
    Группирует файлы по наибольшим общим префиксам длиной >= min_prefix_len.
    Остальные попадают в misc/.
    """
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    files.sort()

    groups = {}
    used = set()

    for i in range(len(files)):
        if files[i] in used:
            continue
        group = [files[i]]
        for j in range(i + 1, len(files)):
            pref = longest_common_prefix(files[i], files[j])
            if len(pref) >= min_prefix_len:
                group.append(files[j])
                used.add(files[j])
        groups[files[i]] = group
        used.add(files[i])

    # раскладываем по папкам
    for leader, group in groups.items():
        if len(group) == 1:
            folder_name = "misc"
        else:
            folder_name = os.path.splitext(leader)[0][:min_prefix_len] + "_group"
        folder_path = os.path.join(directory, folder_name)
        os.makedirs(folder_path, exist_ok=True)

        for f in group:
            src = os.path.join(directory, f)
            dst = os.path.join(folder_path, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)

    print("Файлы разложены по папкам.")

if __name__ == "__main__":
    sort_files_into_folders("/root/TNDP_learning/output_routes", min_prefix_len=3)


Файлы разложены по папкам.


In [11]:
import os

def list_files(directory: str):
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    for f in files:
        print(f)

if __name__ == "__main__":
    list_files("/root/TNDP_learning/output_routes")  # замени на свою папку


neural_bco_exp_mumford0_pp_0_op_0_cp_1_generated_routes.pkl
neural_bco_exp_mumford0_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
neural_bco_exp_mandl_pp_0.5_op_0.5_cp_0_generated_routes.pkl
nn_construction_exp_mandl_pp_0_op_1_cp_0_starting_routes.pkl
nn_construction_mandl_weighted_connectivity_eval_pp_1_op_0_cp_0_routes.pkl
nn_construction_mandl_eval_pp_0.2_op_0.5_cp_0.5_routes.pkl
nn_construction_exp_mumford1_pp_1_op_0_cp_0_starting_routes.pkl
neural_bco_exp_multimodal_mandlbus_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
neural_bco_exp_mumford0_pp_0.5_op_0.5_cp_0_generated_routes.pkl
nn_construction_exp_mumford1_pp_0.5_op_0.5_cp_0_starting_routes.pkl
neural_bco_exp_mandl_pp_0.5_op_0_cp_0.5_generated_routes.pkl
nn_construction_exp_mumford1_pp_0.5_op_0_cp_0.5_starting_routes.pkl
nn_construction_exp_dt1_rt05_ct0_routes.pkl
nn_construction_exp_mumford1_pp_0_op_0.5_cp_0.5_starting_routes.pkl
nn_construction_exp_mumford1_pp_0.33_op_0.33_cp_0.33_starting_routes.pkl
nn_construction_mandl_eva

In [10]:
import os
import shutil

def flatten_directory(root_dir: str):
    """
    Переносит все файлы из подпапок в root_dir и удаляет пустые папки.
    """
    for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
        if dirpath == root_dir:
            continue  # саму корневую папку не трогаем
        for filename in filenames:
            src = os.path.join(dirpath, filename)
            dst = os.path.join(root_dir, filename)

            # если в корне уже есть файл с таким именем — добавим суффикс
            if os.path.exists(dst):
                name, ext = os.path.splitext(filename)
                k = 1
                while True:
                    new_name = f"{name}_{k}{ext}"
                    dst = os.path.join(root_dir, new_name)
                    if not os.path.exists(dst):
                        break
                    k += 1

            shutil.move(src, dst)

        # удаляем пустую папку
        if not os.listdir(dirpath):
            os.rmdir(dirpath)

    print(f"Все файлы перенесены в {root_dir}, подпапки удалены.")

if __name__ == "__main__":
    flatten_directory("/root/TNDP_learning/output_routes")


Все файлы перенесены в /root/TNDP_learning/output_routes, подпапки удалены.


In [13]:
import os
import shutil
from collections import defaultdict
from pathlib import Path

def extract_base_name(filename):
    """
    Извлекает базовое название файла для группировки
    """
    # Убираем расширение .pkl
    name = filename.replace('.pkl', '')
    
    # Определяем основные паттерны для группировки
    if name.startswith('neural_bco_exp_'):
        # Для neural_bco_exp файлов извлекаем тип эксперимента
        parts = name.split('_')
        if len(parts) >= 4:
            return f"neural_bco_{parts[3]}"  # например: neural_bco_mandl
    
    elif name.startswith('nn_construction_exp_'):
        # Для nn_construction_exp файлов извлекаем тип эксперимента
        parts = name.split('_')
        if len(parts) >= 4:
            return f"nn_construction_{parts[3]}"  # например: nn_construction_mandl
    
    elif name.startswith('nn_construction_mandl_eval_'):
        # Все evaluation файлы в одну папку
        return "nn_construction_mandl_eval"
    
    elif name.startswith('nn_construction_mandl_weighted_connectivity_'):
        return "nn_construction_mandl_weighted_connectivity"
    
    elif name.startswith('nn_construction_Kemerovo_'):
        return "nn_construction_Kemerovo"
    
    elif name.startswith('neural_bco_Kemerovo_'):
        return "neural_bco_Kemerovo"
    
    elif 'multimodal' in name:
        # Группируем все multimodal файлы
        if 'neural_bco' in name:
            return "neural_bco_multimodal"
        elif 'nn_construction' in name:
            return "nn_construction_multimodal"
    
    # Для остальных файлов извлекаем первые два компонента
    parts = name.split('_')
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1]}"
    
    return "other"

def sort_files_by_name(source_directory):
    """
    Сортирует файлы по папкам на основе их названий
    """
    source_path = Path(source_directory)
    
    if not source_path.exists():
        print(f"Директория {source_directory} не существует!")
        return
    
    # Собираем все .pkl файлы
    pkl_files = list(source_path.glob("*.pkl"))
    
    if not pkl_files:
        print("В директории не найдено .pkl файлов!")
        return
    
    # Группируем файлы по базовым названиям
    file_groups = defaultdict(list)
    
    for file_path in pkl_files:
        base_name = extract_base_name(file_path.name)
        file_groups[base_name].append(file_path)
    
    print(f"Найдено {len(pkl_files)} файлов")
    print(f"Будет создано {len(file_groups)} папок")
    print("\nГруппировка:")
    
    for group_name, files in file_groups.items():
        print(f"\n{group_name} ({len(files)} файлов):")
        for file in files[:3]:  # Показываем первые 3 файла
            print(f"  - {file.name}")
        if len(files) > 3:
            print(f"  ... и еще {len(files) - 3} файлов")
    
    # Спрашиваем подтверждение
    response = input("\nПродолжить создание папок и перемещение файлов? (y/n): ")
    if response.lower() != 'y':
        print("Операция отменена.")
        return
    
    # Создаем папки и перемещаем файлы
    moved_count = 0
    
    for group_name, files in file_groups.items():
        # Создаем папку
        group_dir = source_path / group_name
        group_dir.mkdir(exist_ok=True)
        
        # Перемещаем файлы
        for file_path in files:
            destination = group_dir / file_path.name
            
            # Если файл уже существует в целевой папке, добавляем суффикс
            counter = 1
            original_destination = destination
            while destination.exists():
                stem = original_destination.stem
                suffix = original_destination.suffix
                destination = group_dir / f"{stem}_{counter}{suffix}"
                counter += 1
            
            shutil.move(str(file_path), str(destination))
            moved_count += 1
            print(f"Перемещен: {file_path.name} -> {group_name}/")
    
    print(f"\nГотово! Перемещено {moved_count} файлов в {len(file_groups)} папок.")

def preview_grouping(source_directory):
    """
    Показывает предварительный просмотр группировки без перемещения файлов
    """
    source_path = Path(source_directory)
    
    if not source_path.exists():
        print(f"Директория {source_directory} не существует!")
        return
    
    pkl_files = list(source_path.glob("*.pkl"))
    file_groups = defaultdict(list)
    
    for file_path in pkl_files:
        base_name = extract_base_name(file_path.name)
        file_groups[base_name].append(file_path.name)
    
    print("ПРЕДВАРИТЕЛЬНЫЙ ПРОСМОТР ГРУППИРОВКИ:")
    print("=" * 50)
    
    for group_name in sorted(file_groups.keys()):
        files = file_groups[group_name]
        print(f"\n📁 {group_name} ({len(files)} файлов):")
        for file in files:
            print(f"   • {file}")

if __name__ == "__main__":
    # Укажите путь к вашей папке с файлами
    source_dir = "/root/TNDP_learning/output_routes"
    
    print("\n1. Предварительный просмотр")

    preview_grouping(source_dir)


1. Предварительный просмотр
ПРЕДВАРИТЕЛЬНЫЙ ПРОСМОТР ГРУППИРОВКИ:

📁 neural_bco_Kemerovo (1 файлов):
   • neural_bco_Kemerovo_exp_mandl_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl

📁 neural_bco_mandl (7 файлов):
   • neural_bco_exp_mandl_pp_0.5_op_0.5_cp_0_generated_routes.pkl
   • neural_bco_exp_mandl_pp_0.5_op_0_cp_0.5_generated_routes.pkl
   • neural_bco_exp_mandl_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
   • neural_bco_exp_mandl_pp_0_op_1_cp_0_generated_routes.pkl
   • neural_bco_exp_mandl_pp_1_op_0_cp_0_generated_routes.pkl
   • neural_bco_exp_mandl_pp_0_op_0_cp_1_generated_routes.pkl
   • neural_bco_exp_mandl_pp_0_op_0.5_cp_0.5_generated_routes.pkl

📁 neural_bco_multimodal (3 файлов):
   • neural_bco_exp_multimodal_mandlbus_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
   • neural_bco_exp_multimodal_mandltrolleybus_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
   • neural_bco_exp_multimodal_mandltrolley_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl

📁 neural_bco_mumford0 (7 фа

In [14]:
sort_files_by_name(source_dir)

Найдено 91 файлов
Будет создано 19 папок

Группировка:

neural_bco_mumford0 (7 файлов):
  - neural_bco_exp_mumford0_pp_0_op_0_cp_1_generated_routes.pkl
  - neural_bco_exp_mumford0_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
  - neural_bco_exp_mumford0_pp_0.5_op_0.5_cp_0_generated_routes.pkl
  ... и еще 4 файлов

neural_bco_mandl (7 файлов):
  - neural_bco_exp_mandl_pp_0.5_op_0.5_cp_0_generated_routes.pkl
  - neural_bco_exp_mandl_pp_0.5_op_0_cp_0.5_generated_routes.pkl
  - neural_bco_exp_mandl_pp_0.33_op_0.33_cp_0.33_generated_routes.pkl
  ... и еще 4 файлов

nn_construction_mandl (7 файлов):
  - nn_construction_exp_mandl_pp_0_op_1_cp_0_starting_routes.pkl
  - nn_construction_exp_mandl_pp_1_op_0_cp_0_starting_routes.pkl
  - nn_construction_exp_mandl_pp_0.33_op_0.33_cp_0.33_starting_routes.pkl
  ... и еще 4 файлов

nn_construction_mandl_weighted_connectivity (2 файлов):
  - nn_construction_mandl_weighted_connectivity_eval_pp_1_op_0_cp_0_routes.pkl
  - nn_construction_mandl_weighted_conn